In [1]:
import json
from pathlib import Path
from argparse import ArgumentParser
from dimacs_parser import DimacsParser
from model_timer import Timer

# import numpy as np
input_file = '../input/C208_120.cnf'
path = Path(input_file)
filename = path.name
instance = DimacsParser.parse_cnf_file(input_file)
print(instance, end="")

Number of variables: 1608
Number of clauses: 5278
Variables: {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 45, 50, 51, 52, 58, 64, 65, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 84, 85, 89, 90, 91, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 129, 130, 131, 132, 133, 136, 139, 140, 142, 143, 144, 145, 146, 152, 153, 154, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 25

In [2]:
UNASSIGNED = -1
TRUE = 1
FALSE = 0

num_vars = max(abs(lit) for clause in instance.clauses for lit in clause)
model = [UNASSIGNED] * (num_vars + 1)
clauses = [list(clause) for clause in instance.clauses]

In [3]:
clause = clauses[0]

In [4]:
for lit in clause:
    var = abs(lit)
    val = model[var]

In [5]:
''' 
clause/instance evaluation 
change to return ints rather than strings
'''

def eval_clause(clause, model):
    unassigned = False 
    
    for lit in clause: 
        var = abs(lit) 
        val = model[var] 

        if val == UNASSIGNED: 
            unassigned = True 
        else: 
            if (lit > 0 and val == TRUE) or (lit < 0 and val == FALSE):
                return TRUE 
    
    if unassigned: 
        return UNASSIGNED 
    
    return FALSE 

def eval_instance(clauses, model): 
    all_true = True 
    
    for clause in clauses: 
        clause_value = eval_clause(clause, model) 

        if clause_value == FALSE: 
            return FALSE 
        if clause_value != TRUE: 
            all_true = False 
    if all_true:
        return TRUE 
    
    return UNASSIGNED

In [6]:
''' 
pure symbol 
use list of integers and boolean array, only check unassigned variables
'''

def pure_symbol(clauses, model): 
    num_vars = len(model) - 1
    pure = [0] * (num_vars + 1) 

    for clause in clauses: 
        clause_val = eval_clause(clause, model) 
        if clause_val == TRUE: 
            continue 

        for lit in clause: 
            var = abs(lit) 
            if model[var] != UNASSIGNED or pure[var] == 2: 
                continue 

            if lit > 0: 
                sign = 1 
            else: 
                sign = -1 

            if pure[var] == 0: 
                pure[var] = sign 
            elif pure[var] != sign: 
                pure[var] = 2 

    vars_list = [] 
    vals_list = [] 
    for var in range(1, num_vars + 1): 
        if pure[var] == 1: 
            vars_list.append(var) 
            vals_list.append(TRUE) 
        elif pure[var] == -1: 
            vars_list.append(var) 
            vals_list.append(FALSE) 
    
    return vars_list, vals_list 

In [7]:
''' 
unit clause 
'''

def unit_clause(clauses, model):
    assigned = []
    changed = True
    while changed:
        changed = False
        for clause in clauses:
            val = eval_clause(clause, model)
            if val == FALSE:
                # conflict
                for var in assigned:
                    model[var] = UNASSIGNED
                return False
            if val == TRUE:
                continue

            unassigned_lits = [lit for lit in clause if model[abs(lit)] == UNASSIGNED]
            if len(unassigned_lits) == 1:
                lit = unassigned_lits[0]
                var = abs(lit)
                model[var] = TRUE if lit > 0 else FALSE
                assigned.append(var)
                changed = True
    return assigned


In [20]:
'''
branching 
'''

# def branch_var(clauses, model): 
#     num_vars = len(model) - 1
#     vars = [0] * (num_vars + 1)
#     poss = [0] * (num_vars + 1)
#     negs = [0] * (num_vars + 1) 

#     for clause in clauses: 
#         clause_val = eval_clause(clause, model) 
#         if clause_val == TRUE: continue 

#         for lit in clause: 
#             var = abs(lit) 
#             if model[var] != UNASSIGNED: continue 

#             vars[var] += 1
#             if lit > 0: 
#                 poss[var] += 1 
#             else: 
#                 negs[var] += 1 

#     best_var = None 
#     best_score = -1 
#     for var in range(1, num_vars + 1): 
#         if model[var] == UNASSIGNED and vars[var] > best_score: 
#             best_var = var 
#             best_score = vars[var] 
    
#     if best_var is None: 
#         return None, None 

#     sign = poss[best_var] >= negs[best_var]
#     return best_var, sign

def branch_var(clauses, model):
    num_vars=len(model)-1
    min_size=float('inf')
    pos_count = [0] * (num_vars+1)
    neg_count = [0] * (num_vars+1)

    for clause in clauses:
        clause_val = eval_clause(clause, model) 
        if clause_val==TRUE:
            continue 
        unassigned=[] 

        for lit in clause: 
            var=abs(lit) 
            if model[var]==UNASSIGNED:
                unassigned.append(lit) 

        if len(unassigned)==0: 
            continue 
        
        size = len(unassigned) 
        if size < min_size:
            min_size=size 
            pos_count = [0] * (num_vars + 1)
            neg_count = [0] * (num_vars + 1) 

        if size == min_size: 
            for lit in unassigned:
                var = abs(lit) 
                if lit > 0: 
                    pos_count[var] += 1
                else: 
                    neg_count[var] += 1 
        
    best_score = -1
    best_vars = [] 

    for var in range(1, num_vars+1): 
        if model[var] != UNASSIGNED:
            continue 

        score = pos_count[var] + neg_count[var] 

        if score > best_score: 
            best_score = score 
            best_vars = [var] 
        elif score == best_score and score > 0: 
            best_vars.append(var) 

    if not best_vars: 
        return None, None 

    best_var = random.choice(best_vars) 
    sign = pos_count[best_var] > neg_count[best_var]
    return best_var, sign

In [21]:
def unassigned(model):
    for var in range(1, len(model)):
        if model[var] == UNASSIGNED:
            return var
    return None


In [22]:
input_file = '../toy_solveable.cnf'
path = Path(input_file)
filename = path.name
instance = DimacsParser.parse_cnf_file(input_file)
print(instance, end="")

UNASSIGNED = -1
TRUE = 1
FALSE = 0

num_vars = max(abs(lit) for clause in instance.clauses for lit in clause)
model = [UNASSIGNED] * (num_vars + 1)
clauses = [list(clause) for clause in instance.clauses]

Number of variables: 26
Number of clauses: 130
Variables: {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26}
Clause 0: {-23, 21, -1}
Clause 1: {-23, -13, -18}
Clause 2: {24, 26, -4}
Clause 3: {16, -15, 13}
Clause 4: {9, -4, 5}
Clause 5: {-8, -7, -10}
Clause 6: {25, 12, -3}
Clause 7: {-15, -21, 17}
Clause 8: {18, -1, -17}
Clause 9: {8, -4, 22}
Clause 10: {24, 16, -2}
Clause 11: {-19, 22, -17}
Clause 12: {-23, 11, 25}
Clause 13: {1, 5, -3}
Clause 14: {23, -12, 7}
Clause 15: {-15, -6, 13}
Clause 16: {-12, 13, -17}
Clause 17: {9, -5, -19}
Clause 18: {-25, 15}
Clause 19: {2, 26, -22}
Clause 20: {21, -2, -17}
Clause 21: {2, 5, -9}
Clause 22: {26, -4, -2}
Clause 23: {-7, 1, -9}
Clause 24: {-16, -24, 21}
Clause 25: {-6, -22, -17}
Clause 26: {12, 14, 7}
Clause 27: {9, 5, -10}
Clause 28: {18, -13, 4}
Clause 29: {17, 2, 7}
Clause 30: {12, -18}
Clause 31: {-22, 3, 21}
Clause 32: {10, -4, 15}
Clause 33: {24, 12, 5}
Clause 34: {18, -26, 7}
Clause 35: {-16

In [23]:
import random 
def dpll(clauses, model): 
    status = eval_instance(clauses, model) 

    if status == TRUE:
        return model 
    if status == FALSE: 
        return None 
    
    assigned_vars = unit_clause(clauses, model) 
    if assigned_vars is False: 
        return None 
    
    pure_vars, pure_vals = pure_symbol(clauses, model) 
    for var, val in zip(pure_vars, pure_vals): 
        model[var] = val 
    
    var, sign = branch_var(clauses, model)
    if var is None: 
        return model if eval_instance(clauses, model) == TRUE else None
    
    new_model = model.copy()
    new_model[var] = TRUE if sign else FALSE 
    result = dpll(clauses, new_model)
    if result is not None:
        model[:] = new_model
        return result 
    
    new_model = model.copy() 
    new_model[var] = FALSE if sign else TRUE
    result = dpll(clauses, new_model)
    if result is not None:
        model[:] = new_model
        return result
    
    model[var] = UNASSIGNED 
    for v in assigned_vars:
        model[v] = UNASSIGNED
    for v, val in zip(pure_vars, pure_vals): 
        model[v] = UNASSIGNED

    return None 

In [24]:
dpll(clauses, model)

[-1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0]